# Aim
Use the established SignalStreamer to stream 1090 data and continuously store aircraft datacoverage as possible

Assess a range of different parameterisations with the intent of trying to have as close to continuous 


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import pandas as pd
import numpy as np
import scipy
import rs1090
import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px
from copy import copy
import plotly.graph_objects as go

import cosmosdr
from cosmosdr.plotting import basic_plot
import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
import cosmosdr.plane_sailing as planes

from cosmosdr.plotting import create_base_figure

import structlog
logger = structlog.get_logger()

try:
    sdr.close()
except:
    pass

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2.4e6


# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us_after_upsampling = int(round(target_sr/1e6))  # 12

In [ ]:
s_acq.streamer.stop_stream()

n_reads_per_acquisition = int(32 * 1)
n_samples_per_read = int(4096 * 1)

s_acq.streamer.start_stream(
    center_freq=center_freq, 
    sample_rate=sample_rate,
    n_reads_per_acquisition=n_reads_per_acquisition,
    n_samples_per_read=n_samples_per_read, 
    sleep_length_s=0.05,
    sdr_gain="auto"
)


In [ ]:
%%time

n = 100

attempts = []
hits = 0
aircraft_info = []

for i in range(n):
    time.sleep(0.1)
    signal = s_acq.streamer.get_current_signal()

    # tenth_highest_pulse_per_read = np.sort(np.abs(signal))[:,-10]
    # index_of_strongest_signal = tenth_highest_pulse_per_read.argmax()
    index_of_strongest_signal = s_acq.get_index_of_highest_peak(signal, verbose=True)
    iq = signal[index_of_strongest_signal]

    # Only do the expensive signal processing on the strongest read
    iq_mag = planes.preprocess_iq(iq, orig_sr=sample_rate, target_sr=target_sr)
    try:
        iq_mag_bin = planes.convert_to_binary(iq_mag)
    except:
        continue

    adsb = planes.decode_to_adsb(iq_mag_bin)
    if adsb is not None:
        hex_string = planes.convert_adsb_binary_to_hex(adsb)
        decoded = rs1090.decode(hex_string)
        if decoded:
            logger.info("!Woohoo decoded!")
            hits += 1
            aircraft_info.append(decoded)

    else:
        hex_string = None

    attempts.append([tenth_highest_pulse_per_read.max(), adsb, hex_string, decoded])
    

print("n_reads_per_acquisition: ", n_reads_per_acquisition)
print("n_samples_per_read: ", n_samples_per_read)
print("hits: ", hits)

In [ ]:
n_reads_per_acquisition:  1
n_samples_per_read:  131072
hits:  16
CPU times: user 9.83 s, sys: 57.4 ms, total: 9.88 s
Wall time: 21.8 s

In [ ]:
n_reads_per_acquisition:  8
n_samples_per_read:  16384
hits:  13
CPU times: user 4.69 s, sys: 236 ms, total: 4.93 s
Wall time: 17 s

In [ ]:
n_reads_per_acquisition:  16
n_samples_per_read:  8192
hits:  14
CPU times: user 4.89 s, sys: 369 ms, total: 5.26 s
Wall time: 17 s

n_reads_per_acquisition:  32
n_samples_per_read:  4096
hits:  19, 6, 0 (then got 20 when I make the acq happen faster than the check
CPU times: user 5.37 s, sys: 554 ms, total: 5.93 s
Wall time: 16.7 s

In [ ]:
n_reads_per_acquisition:  64
n_samples_per_read:  2048
hits:  12, 6
CPU times: user 6.16 s, sys: 874 ms, total: 7.04 s
Wall time: 16.4 s

In [ ]:
n_reads_per_acquisition:  128
n_samples_per_read:  1024
hits:  13
CPU times: user 5.92 s, sys: 940 ms, total: 6.86 s
Wall time: 16.1 s

In [ ]:
aircraft_info